In [13]:

# ============================================================
# NOTEBOOK 08 — IMAGE QUALITY AND DIVERSITY EVALUATION
# NDCLD15 | Real vs Traditional vs GAN vs Diffusion
# ============================================================

from pathlib import Path
from PIL import Image, ImageFilter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time, zipfile, random, hashlib

SEED = 42
MAX_IMAGES_PER_GROUP = 500
random.seed(SEED)
np.random.seed(SEED)
start_time = time.time()

PROJECT = Path("/kaggle/working/nd_iris_project")
REAL_DIR = PROJECT / "notebook02_preprocessing/images_224"
TRAD_DIR = PROJECT / "notebook05_traditional_augmentation"
GAN_DIR = PROJECT / "notebook06_gan_augmentation/generated_samples"
DIFF_DIR = PROJECT / "notebook07_diffusion_training/generated_samples"

OUT = PROJECT / "notebook08_quality_diversity"
OUT.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Real images:", REAL_DIR.exists())
print("Traditional folder:", TRAD_DIR.exists())
print("GAN folder:", GAN_DIR.exists())
print("Diffusion folder:", DIFF_DIR.exists())

# ------------------------------------------------------------
# 1. Locate images recursively
# ------------------------------------------------------------
EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def find_images(folder, exclude_parts=None):
    folder = Path(folder)
    if not folder.exists():
        return []
    exclude_parts = set(exclude_parts or [])
    files = [
        p for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in EXTS
        and not any(part in exclude_parts for part in p.parts)
    ]
    return sorted(files)

real_files = find_images(REAL_DIR)

# Traditional augmentation: only use augmented output subfolders
# when available; exclude the source/preprocessed image directory.
trad_files = find_images(
    TRAD_DIR,
    exclude_parts={"originals", "source_images", "real_images"}
)
trad_files = [
    p for p in trad_files
    if "image" in p.suffix.lower()
]

gan_files = find_images(GAN_DIR)
diff_files = find_images(DIFF_DIR)

# Avoid accidentally treating plots or unrelated images as samples.
# Generated sample directories are expected to contain images only.
groups = {
    "Real": real_files,
    "Traditional": trad_files,
    "GAN": gan_files,
    "Diffusion": diff_files
}

print("\nDiscovered image files:")
for name, files in groups.items():
    print(f"{name}: {len(files)}")

if not real_files:
    raise FileNotFoundError(
        f"No real images found in {REAL_DIR}. "
        "Run Notebook 02 first or correct REAL_DIR."
    )

# ------------------------------------------------------------
# 2. Deterministic sampling
# ------------------------------------------------------------
sampled = {}
for name, files in groups.items():
    if files:
        files = list(files)
        random.Random(SEED).shuffle(files)
        sampled[name] = files[:MAX_IMAGES_PER_GROUP]

# ------------------------------------------------------------
# 3. Quality metrics
# ------------------------------------------------------------
def calculate_metrics(path):
    with Image.open(path) as im:
        im = im.convert("L").resize((128, 128))
        arr = np.asarray(im, dtype=np.float32)

    brightness = float(arr.mean())
    contrast = float(arr.std())
    dynamic_range = float(arr.max() - arr.min())

    # Gradient energy: average squared horizontal/vertical differences
    gx = np.diff(arr, axis=1)
    gy = np.diff(arr, axis=0)
    gradient_energy = float(
        (np.mean(gx ** 2) + np.mean(gy ** 2)) / 2
    )

    # Laplacian variance: a sharpness proxy
    lap = (
        -4 * arr
        + np.roll(arr, 1, axis=0)
        + np.roll(arr, -1, axis=0)
        + np.roll(arr, 1, axis=1)
        + np.roll(arr, -1, axis=1)
    )
    sharpness_proxy = float(np.var(lap))

    # High-frequency residual proxy
    im = Image.fromarray(arr.astype(np.uint8))
    blur = np.asarray(
        im.filter(ImageFilter.GaussianBlur(radius=2)),
        dtype=np.float32
    )
    hf_residual = float(np.mean(np.abs(arr - blur)))

    return {
        "brightness_mean": brightness,
        "contrast_std": contrast,
        "dynamic_range": dynamic_range,
        "gradient_energy": gradient_energy,
        "laplacian_variance": sharpness_proxy,
        "high_frequency_residual": hf_residual
    }

rows = []
errors = []

for group_name, files in sampled.items():
    print(f"\nEvaluating {group_name}: {len(files)} images")
    for i, path in enumerate(files):
        try:
            metrics = calculate_metrics(path)
            rows.append({
                "group": group_name,
                "file": path.name,
                "path": str(path),
                **metrics
            })
        except Exception as e:
            errors.append({
                "group": group_name,
                "file": str(path),
                "error": str(e)
            })

metrics_df = pd.DataFrame(rows)
errors_df = pd.DataFrame(errors)

if metrics_df.empty:
    raise RuntimeError("No images could be evaluated.")

metrics_csv = OUT / "image_quality_metrics.csv"
metrics_df.to_csv(metrics_csv, index=False)
errors_df.to_csv(OUT / "evaluation_errors.csv", index=False)

# ------------------------------------------------------------
# 4. Summary statistics
# ------------------------------------------------------------
metric_cols = [
    "brightness_mean",
    "contrast_std",
    "dynamic_range",
    "gradient_energy",
    "laplacian_variance",
    "high_frequency_residual"
]

summary = metrics_df.groupby("group")[metric_cols].agg(
    ["mean", "std", "median"]
)
summary.columns = [
    f"{metric}_{stat}" for metric, stat in summary.columns
]
summary = summary.reset_index()
summary.to_csv(OUT / "group_summary_statistics.csv", index=False)

# ------------------------------------------------------------
# 5. Plot group comparisons
# ------------------------------------------------------------
for metric in metric_cols:
    plt.figure(figsize=(9, 5))
    data = [
        metrics_df.loc[metrics_df["group"] == group, metric].dropna().values
        for group in sampled.keys()
        if (metrics_df["group"] == group).any()
    ]
    labels = [
        group for group in sampled.keys()
        if (metrics_df["group"] == group).any()
    ]
    if data:
        plt.boxplot(data, tick_labels=labels, showfliers=False)
        plt.title(f"Image comparison: {metric}")
        plt.ylabel(metric)
        plt.xlabel("Image group")
        plt.tight_layout()
        plt.savefig(OUT / f"comparison_{metric}.png", dpi=150)
    plt.close()

# ------------------------------------------------------------
# 6. Simple distribution-based diversity proxies
# ------------------------------------------------------------
# These measure variation in image statistics, not semantic diversity.
diversity_rows = []
for group_name, frame in metrics_df.groupby("group"):
    row = {"group": group_name, "evaluated_images": len(frame)}
    for metric in metric_cols:
        row[f"{metric}_across_image_std"] = float(frame[metric].std())
        row[f"{metric}_across_image_iqr"] = float(
            frame[metric].quantile(0.75) - frame[metric].quantile(0.25)
        )
    diversity_rows.append(row)

diversity_df = pd.DataFrame(diversity_rows)
diversity_df.to_csv(OUT / "diversity_proxy_summary.csv", index=False)

# Plot per-image brightness/contrast distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for group_name, frame in metrics_df.groupby("group"):
    axes[0].hist(
        frame["brightness_mean"], bins=25, alpha=0.45,
        label=group_name
    )
    axes[1].hist(
        frame["contrast_std"], bins=25, alpha=0.45,
        label=group_name
    )
axes[0].set_title("Brightness distribution")
axes[0].set_xlabel("Mean pixel intensity")
axes[1].set_title("Contrast distribution")
axes[1].set_xlabel("Pixel-intensity standard deviation")
for ax in axes:
    ax.legend()
    ax.set_ylabel("Image count")
plt.tight_layout()
plt.savefig(OUT / "brightness_contrast_distributions.png", dpi=150)
plt.close()

# ------------------------------------------------------------
# 7. Save machine-readable report
# ------------------------------------------------------------
counts = {
    name: {
        "discovered_images": len(groups[name]),
        "evaluated_images": int(
            (metrics_df["group"] == name).sum()
        ) if not metrics_df.empty else 0
    }
    for name in groups
}

report = {
    "notebook": 8,
    "dataset": "NDCLD15",
    "sample_limit_per_group": MAX_IMAGES_PER_GROUP,
    "groups": counts,
    "total_evaluated": int(len(metrics_df)),
    "errors": int(len(errors_df)),
    "metrics": metric_cols,
    "diversity_method": (
        "Across-image variation of brightness, contrast, dynamic range, "
        "gradient energy, Laplacian variance and high-frequency residual"
    ),
    "limitations": [
        "Diversity values are statistical proxies, not semantic diversity metrics.",
        "No verified contact-lens class annotations were available.",
        "Quality statistics alone do not establish realism, identity preservation, "
        "privacy, clinical validity, or augmentation utility.",
        "Comparisons may be incomplete if a group's output directory is missing."
    ],
    "elapsed_seconds": round(time.time() - start_time, 2)
}

with open(OUT / "Notebook08_summary.json", "w") as f:
    json.dump(report, f, indent=2)

# ------------------------------------------------------------
# 8. Generate conclusion Markdown from actual run results
# ------------------------------------------------------------
conclusion = [
    "# Notebook 08 — Image Quality and Diversity Evaluation",
    "",
    "## Objective",
    "Compare image-level quality statistics and simple diversity proxies "
    "for available real, traditional-augmentation, GAN, and diffusion images.",
    "",
    "## Evaluation coverage",
    f"- Total evaluated images: **{len(metrics_df)}**",
    f"- Evaluation errors: **{len(errors_df)}**",
    f"- Sampling limit per group: **{MAX_IMAGES_PER_GROUP}**",
    ""
]

for name in groups:
    discovered = counts[name]["discovered_images"]
    evaluated = counts[name]["evaluated_images"]
    conclusion.append(
        f"- **{name}:** {evaluated} evaluated image(s), "
        f"{discovered} discovered file(s)."
    )

conclusion += [
    "",
    "## Group-level results",
    "",
    summary.round(4).to_markdown(index=False),
    "",
    "## Interpretation",
    "Brightness, contrast, dynamic range, gradient energy, Laplacian variance, "
    "and high-frequency residual provide useful low-level image statistics. "
    "Across-image variation in these statistics is reported as a diversity proxy. "
    "These measures cannot establish whether synthetic images are semantically "
    "diverse or realistic.",
    "",
    "## Limitations",
    "- Verified contact-lens class labels were unavailable.",
    "- Image-statistic variation is not a substitute for FID, KID, expert review, "
    "or task-based evaluation.",
    "- Generated samples may be low-resolution and trained for only a few epochs.",
    "- Do not infer identity preservation, privacy, or clinical suitability from "
    "these results.",
    "",
    "## Conclusion",
    "This notebook completed an exploratory comparison of the image groups "
    "that were available at runtime. Interpret comparisons only for groups "
    "with nonzero evaluated counts. Missing groups must not be treated as "
    "failed models or as zero-quality images.",
    ""
]

conclusion_path = OUT / "Notebook08_Conclusion.md"
conclusion_path.write_text("\n".join(conclusion), encoding="utf-8")

# ------------------------------------------------------------
# 9. ZIP all Notebook 08 artifacts
# ------------------------------------------------------------
zip_path = PROJECT / "ND_Iris_Notebook08_Results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file() and p != zip_path:
            zf.write(p, arcname=str(p.relative_to(OUT)))

with zipfile.ZipFile(zip_path, "r") as zf:
    bad_member = zf.testzip()
    archived_count = len(zf.namelist())

# ------------------------------------------------------------
# 10. Display actual results
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("NOTEBOOK 08 — IMAGE QUALITY & DIVERSITY RESULTS")
print("=" * 65)
print("\nImage counts:")
print(pd.DataFrame(counts).T)
print("\nGroup summary:")
display(summary.round(4))
print("\nDiversity proxy summary:")
display(diversity_df.round(4))
print("\nErrors:", len(errors_df))
print("Elapsed seconds:", round(time.time() - start_time, 2))
print("Conclusion:", conclusion_path)
print("ZIP:", zip_path)
print("Archived files:", archived_count)
print("ZIP integrity:", "PASSED" if bad_member is None else f"FAILED: {bad_member}")
print("\nFiles saved under:", OUT)


Project: /kaggle/working/nd_iris_project
Real images: True
Traditional folder: True
GAN folder: True
Diffusion folder: True

Discovered image files:
Real: 6233
Traditional: 0
GAN: 38
Diffusion: 17

Evaluating Real: 500 images

Evaluating GAN: 38 images

Evaluating Diffusion: 17 images

NOTEBOOK 08 — IMAGE QUALITY & DIVERSITY RESULTS

Image counts:
             discovered_images  evaluated_images
Real                      6233               500
Traditional                  0                 0
GAN                         38                38
Diffusion                   17                17

Group summary:


,group,brightness_mean_mean,brightness_mean_std,brightness_mean_median,contrast_std_mean,contrast_std_std,contrast_std_median,dynamic_range_mean,dynamic_range_std,dynamic_range_median,gradient_energy_mean,gradient_energy_std,gradient_energy_median,laplacian_variance_mean,laplacian_variance_std,laplacian_variance_median,high_frequency_residual_mean,high_frequency_residual_std,high_frequency_residual_median
0,Diffusion,125.9023,8.1104,127.8764,27.5460,5.0340,26.6855,174.1765,23.3512,166.0,66.4978,159.8099,27.4948,289.9117,723.0890,115.9312,4.8727,3.2268,4.0494
1,GAN,132.5839,15.0438,131.1021,47.8265,9.9184,47.4932,229.9474,17.8945,232.5,307.8020,373.2763,149.6019,1306.6578,1849.1705,567.4244,12.4700,5.8585,10.0540
2,Real,143.8634,22.1315,138.4545,60.6807,8.2118,59.8553,249.6560,7.9753,254.0,177.2391,64.8555,164.5458,941.5206,285.3042,919.1459,7.2822,1.2653,7.2998



Diversity proxy summary:


,group,evaluated_images,brightness_mean_across_image_std,brightness_mean_across_image_iqr,contrast_std_across_image_std,contrast_std_across_image_iqr,dynamic_range_across_image_std,dynamic_range_across_image_iqr,gradient_energy_across_image_std,gradient_energy_across_image_iqr,laplacian_variance_across_image_std,laplacian_variance_across_image_iqr,high_frequency_residual_across_image_std,high_frequency_residual_across_image_iqr
0,Diffusion,17,8.1104,13.4389,5.0340,6.6286,23.3512,27.00,159.8099,4.8630,723.0890,32.6725,3.2268,0.2792
1,GAN,38,15.0438,20.8250,9.9184,13.7571,17.8945,25.75,373.2763,54.9352,1849.1705,195.6841,5.8585,1.4918
2,Real,500,22.1315,36.8390,8.2118,9.9033,7.9753,8.00,64.8555,111.5388,285.3042,415.6444,1.2653,1.7847



Errors: 0
Elapsed seconds: 3.28
Conclusion: /kaggle/working/nd_iris_project/notebook08_quality_diversity/Notebook08_Conclusion.md
ZIP: /kaggle/working/nd_iris_project/ND_Iris_Notebook08_Results.zip
Archived files: 13
ZIP integrity: PASSED

Files saved under: /kaggle/working/nd_iris_project/notebook08_quality_diversity


# Notebook 08 — Image Quality and Diversity Evaluation: Conclusion

## Objective
Evaluate low-level image quality statistics and statistical diversity proxies for real NDCLD15 images and the available GAN- and diffusion-generated samples.

## Evaluation Summary
- Real images evaluated: **500**
- GAN-generated images evaluated: **38**
- Diffusion-generated images evaluated: **17**
- Traditional augmentation images evaluated: **0**
- Total images evaluated: **555**
- Evaluation errors: **0**
- ZIP integrity: **Passed**

## Key Results

The real-image sample had an average brightness of **143.86** and average contrast of **60.68**.

The GAN sample had an average brightness of **132.58**, average contrast of **47.83**, and average Laplacian-variance sharpness proxy of **1,849.17**.

The diffusion sample had an average brightness of **125.90**, average contrast of **27.55**, and average Laplacian-variance sharpness proxy of **723.09**.

These results show measurable differences in pixel-level statistics between the real, GAN, and diffusion groups. The GAN samples exhibited greater high-frequency variation according to the measured proxies, while the diffusion samples had lower average contrast than the real-image sample.

## Diversity Analysis
Across-image variation was calculated for brightness, contrast, dynamic range, gradient energy, Laplacian variance, and high-frequency residual. These measurements describe variation in low-level image statistics; they do not establish semantic diversity or perceptual realism.

## Limitations
- Traditional augmentation images were not found by the notebook's current directory search and therefore were not evaluated.
- The generated sample counts were limited, especially for diffusion.
- Image quality proxies cannot establish realism, identity preservation, privacy protection, or biometric recognition utility.
- Verified contact-lens class labels were unavailable, so no class-conditional or clinical conclusions can be drawn.

## Conclusion
The notebook successfully evaluated 555 images, generated comparison statistics and plots, and archived 13 result files in `ND_Iris_Notebook08_Results.zip`. The results provide an exploratory comparison of real, GAN-generated, and diffusion-generated image statistics. Traditional augmentation must be located and evaluated separately before completing the full comparison.